[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/superfluid/superfluid.ipynb)

# Two-Fluid Hydrodynamics of Superfluid Helium

This example simulates second sound and the fountain effect in superfluid helium across a triangulated cavity divided by a porous powder wall. Below about 2.17 kelvin, liquid helium behaves as two interpenetrating fluids, an inviscid superfluid component carrying mass with zero entropy and a normal fluid component carrying the remaining mass and all of the entropy. Rather than diffusing, a local heat input drives normal fluid outward while drawing superfluid inward, propagating at roughly twenty metres per second as an undamped temperature wave called second sound. Across a porous plug that clamps the normal fluid while admitting the superfluid, heating one chamber draws superfluid through the plug and builds a steady pressure head known as the fountain effect. [Donnelly (2009)](https://doi.org/10.1063/1.3248499) traces how Tisza's two-fluid model and Landau's theory predicted second sound, and the material properties used here come from the reference tables of [Donnelly and Barenghi (1998)](https://doi.org/10.1063/1.556028).

Joining the projective plane with two degenerate material directions for mass and entropy places cavity vertices at unit projective weight. The term ***parcel*** here means the mass and entropy carried per unit volume by a fluid component along those two material directions. Pairing the thermodynamic potentials with the total liquid parcel `potentials & liquid` evaluates the pressure perturbation directly from the Gibbs–Duhem relation, while wedging minus the spatial gradient of the potentials with the potentials forms the drive `-gradient ^ potentials` whose commutator `drive.commutator(parcel)` accelerates each component according to the mass and entropy its parcel carries. Joining each component's velocity with its parcel gives the material flux `velocity & parcel`, and wedging each triangle corner's spatial gradient with that flux `gradient ^ flux` integrates the mass and entropy leaving each vertex's dual cell.

In Landau's two-fluid vector notation, inviscid acceleration splits into two momentum equations coupled through pressure and temperature gradients, $\rho_s\,\partial_t\mathbf v_s = -\frac{\rho_s}{\rho}\nabla p + \rho_s s\nabla T$ and $\rho_n\,\partial_t\mathbf v_n = -\frac{\rho_n}{\rho}\nabla p - \rho_s s\nabla T$, where substituting the Gibbs–Duhem differential $\nabla p = \rho\nabla\mu + \rho s\nabla T$ reduces the two right-hand sides to $-\rho_s\nabla\mu$ and $-\rho_n\nabla\mu - \rho s\nabla T$, alongside separate continuity equations $\partial_t\rho + \nabla\cdot(\rho_s\mathbf v_s + \rho_n\mathbf v_n) = 0$ and $\partial_t(\rho s) + \nabla\cdot(\rho s\,\mathbf v_n) = 0$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.fluids.superfluid import render, scenarios
from examples.util.animation import show_states

# The projective plane for spatial placement, joined with degenerate directions for mass and entropy.
space = Algebra("x+y+w0")
quantities = Algebra("m0s0")
ga = space * quantities
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# Spatial coordinate planes, extensive mass-entropy states, and their conjugate thermodynamic potentials.
Upright = Plane.derive.within(Algebra("x+y+"))                                # Vector+
State = Plane.derive.within(quantities).derive.dual                           # Antivector-
Potential = Plane.derive.within(quantities)                                   # Vector+
# Reference origin at unit projective weight, unit material quantities, and their joint complement for spatial areas.
origin = mv.w.dual()                                                          # [] Point-
mass = mv.m.dual()                                                            # [] State-
entropy = mv.s.dual()                                                         # [] State-
unread = mass & entropy                                                       # [] Antibivector-

## Two-fluid parcels and drives

Because superfluid parcels carry mass with zero entropy, a temperature drive passes over the superfluid and accelerates only the normal fluid through its entropy, whereas a chemical-potential drive accelerates both components in proportion to their mass. Pairing either drive with the combined liquid parcel recovers the force from the fall of pressure. Saturated liquid properties are tabulated from 1.2 to 1.9 kelvin in SI units.

In [ ]:
# Tabulated saturated-liquid properties from 1.2 K to 1.9 K and the index of the 1.8 K operating point.
temperatures, densities, normal_densities = scenarios.temperatures, scenarios.densities, scenarios.normal_densities
first_sounds, heat_capacities, entropies = scenarios.first_sounds, scenarios.heat_capacities, scenarios.entropies
n_temperatures = len(temperatures)
cavity_index = scenarios.cavity_index

# Superfluid parcels carry superfluid mass alone, while normal-fluid parcels carry normal mass and all entropy.
masses = np.stack([densities - normal_densities, normal_densities], axis=-1)   # [n_temperatures, 2]
held_entropies = np.stack([0 * entropies, densities * entropies], axis=-1)     # [n_temperatures, 2]
parcels = masses * mass + held_entropies * entropy                             # [n_temperatures, 2] State-
# Unit spatial gradients of chemical potential and temperature rising along x generate two bivector drives.
drives = stack((-mv.xm, -mv.xs))                                               # [2] Bivector+
forces = drives[:, None].commutator(parcels[cavity_index][None, :])            # [2, 2] Point-

In [ ]:
np.set_printoptions(precision=1, suppress=True)
print("Forces along x from chemical-potential and temperature drives (rows) on superfluid and normal fluid (columns)")
print((mv.x & forces).to_array())

## Thermodynamic stiffness

Thermodynamic stiffness maps perturbations in mass and entropy per unit volume to perturbations in chemical potential and temperature. Compression at fixed specific entropy raises the chemical potential, while heating at fixed density raises the temperature and lowers the chemical potential in proportion to specific entropy. Expressing each process as a symmetric dyad `heating * (heating & State)` enforces Maxwell reciprocity automatically.

In [ ]:
# Unit rise in entropy per unit mass measures entropy added beyond what incoming mass carries at fixed specific entropy.
heating = mv.s - entropies * mv.m                                              # [n_temperatures] Potential+
stiffnesses = (first_sounds ** 2 / densities) * (mv.m * (mv.m & State)) + (
    temperatures / (heat_capacities * densities)
) * (heating * (heating & State))                                              # [n_temperatures] Potential+ <- State-

## Cavity triangulation and simplex gradients

The term ***simplex gradient*** here means the pure spatial plane across a triangle that vanishes along the edge facing a corner and evaluates to one at that corner. The cavity triangulation divides a rectangular enclosure into two open chambers separated by a porous powder wall, with a localized heater inside the wider left chamber. Cavity vertices sit at unit projective weight in the plane, extensive mass and entropy perturbations live on the dual cells surrounding the vertices, and component velocities live on the triangles, with lengths in millimetres and times in milliseconds so speeds remain in metres per second.

In [ ]:
# Cavity triangulation, vertex coordinates, porous-wall triangle mask, and initial Gaussian temperature-pulse profile.
topology, coordinates, powder_triangles, pulse_profile = scenarios.cavity_scene(scenarios.n_columns, scenarios.n_rows)
vertices = (origin + mv(Upright, coordinates).dual()).field()                  # Point-[V]
incidence = topology.incidence[2, 0]                                           # [3] Scalar+[F <- V]
corners = incidence * vertices                                                 # [3] Point-[F]
facing = topology.off[2, 0] * vertices                                         # [2, 3] Point-[F]
sides = facing[0] & facing[1]                                                  # [3] Antibivector-[F]
# Joining all three corners with the material complement gives twice each triangle's signed spatial area.
double_areas = corners[0] & corners[1] & corners[2] & unread                   # Antiscalar-[F]
areas = double_areas.norm() / 2                                                # Scalar+[F]
gradients = ((sides & unread) / (sides & unread & corners)).cast(Upright)      # [3] Plane+[F]
dual_areas = (incidence.adjugate() * (areas / 3)).sum(axis=0)                  # Scalar+[V]
render.cavity_figure(topology, vertices, powder_triangles);

Grey strokes show the cavity triangulation inside the outer walls, with the triangles forming the porous powder wall shaded pale tan.

## Heat pulse and second sound

Without viscosity, thermal conduction, or mutual friction, the two fluids evolve conservatively under their thermodynamic drives and dual-cell continuity. Because wedging a corner's simplex gradient with the flux `gradient ^ flux` pairs with the vertex potentials through the adjugate of the drive operator, kinetic energy gained by the components balances internal energy lost by the dual cells. Clamping the normal fluid inside the porous wall blocks entropy transport while allowing superfluid mass flow. Maxwell symmetry of the dyad stiffness means evaluating `stiffness(entropy)` and `stiffness(liquid)` yields potential planes whose pairings with each state read temperature and pressure perturbations directly. This cell takes about ten seconds to step the cavity and render the animation.

In [ ]:
# Operating stiffness and parcels at 1.8 K, half-Courant time step in milliseconds, and porous-wall mobility mask.
stiffness = stiffnesses[cavity_index]                                          # [] Potential+ <- State-
liquid_parcels = parcels[cavity_index]                                         # [2] State-
dt = scenarios.courant * (scenarios.width / (scenarios.n_columns - 1)) / first_sounds[cavity_index]
n_steps, stride = scenarios.n_steps, scenarios.stride
n_frames = n_steps // stride + 1
normal_clamp = mv.scalar(powder_triangles[:, None] * 1.0).field()              # Scalar+[F]
mobility = (1 - normal_clamp * np.array([0, 1])) / masses[cavity_index]        # [2] Scalar+[F]


def cavity_states(state: State, supply: State) -> Iterator[State]:
    """Yields the mass and entropy perturbations per unit volume (`State-[V]`) every `stride` steps across the cavity."""
    velocities = 0 * ((mv.x + mv.y).dual() * mobility)                         # [2] Point-[F]
    for step in range(n_steps + 1):
        if step % stride == 0:
            yield state
        potentials = stiffness(state)                                          # Potential+[V]
        drive = -(gradients ^ (incidence * potentials)).sum(axis=0)            # Bivector+[F]
        velocities = velocities + dt * mobility * drive.commutator(liquid_parcels)  # [2] Point-[F]
        flux = (velocities & liquid_parcels).sum(axis=0)                       # Antibivector-[F]
        # Dual-cell outflow through each corner's simplex gradient balances external entropy supply.
        outflow = (incidence.adjugate() * (areas * (gradients ^ flux))).sum(axis=0) / dual_areas  # State-[V]
        state = state + dt * (supply - outflow)


# Isochoric entropy perturbation corresponding to the Gaussian millikelvin temperature pulse.
pulse_entropy = pulse_profile * densities[cavity_index] * heat_capacities[cavity_index] / temperatures[cavity_index]  # [V]
initial_pulse = (0 * mass + entropy * pulse_entropy).field()                   # State-[V]
zero_supply = (0 * mass + 0 * entropy * pulse_entropy).field()                 # State-[V]
liquid = liquid_parcels.sum(axis=-1)                                           # [] State-
thermometer = stiffness(entropy)                                               # [] Potential+
barometer = stiffness(liquid)                                                  # [] Potential+
fountain_per_kelvin = densities[cavity_index] * entropies[cavity_index]
temperature_scale = scenarios.pulse_temperature_share * scenarios.pulse_rise
pressure_scale = scenarios.pulse_pressure_share * fountain_per_kelvin * scenarios.pulse_rise
pulse_states = show_states(
    cavity_states(initial_pulse, zero_supply),
    render.cavity_animation,
    topology,
    vertices,
    powder_triangles,
    thermometer,
    barometer,
    temperature_scale,
    pressure_scale,
    render.n_bands,
    name="superfluid_pulse",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=n_frames,
);

Warm red and cool blue bands in the upper panel show positive and negative temperature perturbations, while orange and purple bands in the lower panel show positive and negative pressure perturbations across the grey-bordered porous wall. The initial heat pulse expands as a ring of second sound trailed by a cool wake and reflects off the porous wall. Pressure stays nearly uniform until the thermal ring reaches the porous wall, where superfluid drawn toward the warm face streams through the plug and sets up a pressure oscillation between the chambers. Faint temperature ripples in the far chamber arise where superfluid emerges from the plug into stationary normal fluid without carrying net entropy across the wall.

## Fountain effect across the porous wall

Continuous heating warms the left chamber steadily while the porous wall prevents entropy from escaping into the right chamber. Inside the porous wall only the superfluid moves, accelerated solely by gradients of chemical potential, so mechanical balance across the wall holds once the warm chamber's pressure exceeds the cold chamber's pressure by density times specific entropy times the temperature difference, called London's fountain pressure. Pairing a level chemical potential and a temperature jump with the total liquid parcel recovers that pressure head directly from the Gibbs–Duhem relation, and the chamber pressure oscillates around it as superfluid sloshes through the plug. This cell takes about fifteen seconds to step the cavity and render the animation.

In [ ]:
# Steady entropy supply shaped like the Gaussian pulse so the heated chamber warms at warming_rate on average.
cell_area = (scenarios.width / (scenarios.n_columns - 1)) * (scenarios.height / (scenarios.n_rows - 1))
chamber_area = (scenarios.wall_middle - scenarios.wall_half_thickness) * scenarios.height
heating_rate = pulse_profile / (pulse_profile.sum() * cell_area) * scenarios.warming_rate * chamber_area  # [V]
entropy_supply = heating_rate * densities[cavity_index] * heat_capacities[cavity_index] / temperatures[cavity_index]  # [V]
rest_state = (0 * mass + 0 * entropy * entropy_supply).field()                 # State-[V]
steady_supply = (0 * mass + entropy * entropy_supply).field()                  # State-[V]
times = np.arange(n_frames) * stride * dt                                      # [n_frames]
heater_temperature_scale = scenarios.heater_share * scenarios.warming_rate * times[-1]
heater_states = show_states(
    cavity_states(rest_state, steady_supply),
    render.cavity_animation,
    topology,
    vertices,
    powder_triangles,
    thermometer,
    barometer,
    heater_temperature_scale,
    fountain_per_kelvin * heater_temperature_scale,
    None,
    name="superfluid_heater",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=n_frames,
);

Both panels use the same colour conventions as the pulse animation. Heat accumulates inside the left chamber through reverberating second-sound waves while superfluid inflow raises pressure on the warm side and lowers pressure on the cold side.

In [ ]:
# checks
# Temperature drive passes over the superfluid and pushes normal fluid by its entropy; chemical-potential drive pushes both by mass.
assert (((mv.x & forces) + np.stack([masses[cavity_index], held_entropies[cavity_index]])).abs() < 1e-12).all()
# Pairing with the total liquid parcel recovers the pressure force from the Gibbs-Duhem relation.
assert (((mv.x & drives.commutator(liquid)) + (stack((mv.m, mv.s)) & liquid)).abs() < 1e-12).all()
# Total mass and entropy across the closed cavity remain conserved during the pulse.
initial_total = (dual_areas * pulse_states[0]).sites.sum()                     # [] State-
final_total = (dual_areas * pulse_states[-1]).sites.sum()                      # [] State-
assert ((stack((mv.m, mv.s)) & (final_total - initial_total)).abs() < 1e-10).all()
# Clamping the normal fluid inside the porous wall keeps the far chamber free of added entropy at every frame.
far_mask = mv.scalar((coordinates[:, 0] >= scenarios.wall_middle + scenarios.wall_half_thickness)[:, None] * 1.0).field()
far_entropies = stack([(far_mask * dual_areas * (mv.s & state)).sites.sum() for state in pulse_states])  # [n_frames] Scalar+
initial_entropy = (dual_areas * (mv.s & pulse_states[0])).sites.sum()          # [] Scalar+
assert (far_entropies.abs() < 1e-12 * initial_entropy).all()
# Mean chamber pressure difference oscillates around London's fountain pressure over the second half of steady heating.
hot_mask = mv.scalar((coordinates[:, 0] < scenarios.wall_middle - scenarios.wall_half_thickness)[:, None] * 1.0).field()
cold_mask = mv.scalar((coordinates[:, 0] > scenarios.wall_middle + scenarios.wall_half_thickness)[:, None] * 1.0).field()
chamber_weights = dual_areas * hot_mask / (dual_areas * hot_mask).sites.sum() - dual_areas * cold_mask / (dual_areas * cold_mask).sites.sum()  # Scalar+[V]
pressure_differences = stack([(chamber_weights * (barometer & state)).sites.sum() for state in heater_states])  # [n_frames] Scalar+
fountain_pressures = fountain_per_kelvin * stack(
    [(chamber_weights * (thermometer & state)).sites.sum() for state in heater_states]
)                                                                              # [n_frames] Scalar+
half_frame = n_frames // 2
assert ((pressure_differences[half_frame:].mean() - fountain_pressures[half_frame:].mean()).abs() < 0.15 * fountain_pressures[half_frame:].mean()).all()